# Autonomous AI Data Scientist (AADS) — Google Colab Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uday0214/IOTAxEnigmaHackathon/blob/uday/notebooks/colab_training.ipynb)

**Competition**: Utkrishta 2026 AI/ML Hackathon — Challenge 1 (Budget-Constrained Churn Prediction)

This interactive Google Colab notebook executes the **Autonomous AI Data Scientist** framework:
- **Dual-Brain Supervisor**: Google Gemini 2.5 Flash LLM + Deterministic Mathematical Inference Engine
- **Budget Invariant**: Strict $\le 15$ model fits enforced by atomic `BudgetGuard`
- **Hardware Acceleration**: Automatic NVIDIA GPU detection and CatBoost CUDA acceleration
- **Data Integrity Guard**: SHA-256 cryptographic hash and index verification against data leakage
- **End-to-End Pipeline**: Zero-cost profiling, 15-run feedback loop, sealed holdout evaluation, and submission export

## Step 1: Verify Hardware Accelerator (GPU Check)

Ensure your Google Colab runtime is set to GPU:
1. Go to the menu: **Runtime** -> **Change runtime type**
2. Hardware accelerator: Select **T4 GPU** (or A100 / V100)
3. Click **Save**

In [ ]:
!nvidia-smi

## Step 2: Clone Repository & Setup Working Directory

In [ ]:
import os
import sys

# Clone the repository if not already cloned in /content
repo_name = "IOTAxEnigmaHackathon"
if not os.path.exists(repo_name):
    !git clone -b uday https://github.com/uday0214/IOTAxEnigmaHackathon.git
    %cd IOTAxEnigmaHackathon
else:
    %cd IOTAxEnigmaHackathon
    !git pull

sys.path.insert(0, os.getcwd())
print(f"Current working directory: {os.getcwd()}")

## Step 3: Install Required Dependencies

In [ ]:
!pip install -q -r requirements.txt
print("Dependencies installed successfully!")

## Step 4: Configure Gemini API Key via Google Colab Secrets

The Gemini Supervisor Agent uses Google AI Studio Gemini API (`gemini-2.5-flash`).

### How to set up your key in Colab:
1. Click the **Key icon** (Secrets) in the left sidebar of Google Colab.
2. Click **Add new secret**.
3. Name: `GEMINI_API_KEY`
4. Value: Paste your Gemini API key (from [Google AI Studio](https://aistudio.google.com/)).
5. Toggle **Notebook access** ON.

In [ ]:
import os
from src.utils.environment import get_gemini_api_key

api_key = get_gemini_api_key()
if api_key:
    os.environ["GEMINI_API_KEY"] = api_key
    print("GEMINI_API_KEY detected and active!")
else:
    # Interactive fallback prompt if not in Colab Secrets
    import getpass
    user_key = getpass.getpass("Enter your GEMINI_API_KEY (leave empty to run in offline mock supervisor mode): ")
    if user_key.strip():
        os.environ["GEMINI_API_KEY"] = user_key.strip()
        print("GEMINI_API_KEY configured.")
    else:
        os.environ["MOCK_SUPERVISOR"] = "true"
        print("Running in MOCK_SUPERVISOR mode (deterministic offline simulation).")

## Step 5: Data Preparation & Cryptographic Manifest Generation

Prepares the 80% train-val and 20% sealed holdout partitions and generates the immutable SHA-256 fingerprint registry (`artifacts/dataset_manifest.json`).

In [ ]:
!python scripts/prepare_data.py

## Step 6: Launch the 15-Run Autonomous Training Loop

Runs the end-to-end autonomous optimization loop with GPU acceleration (`--device auto`) and live model fitting (`--live`).
Strictly capped at 15 experiments by the atomic `BudgetGuard`.

In [ ]:
!python scripts/run_autonomous_loop.py --live --device auto --clean-ledger

## Step 7: Telemetry Inspection & Interactive Visualization

Load the JSONL experiment ledger and visualize:
1. **ROC-AUC & PR-AUC Progression** across the 15 runs
2. **Overfitting Index & Loss Trajectory** (Train vs Validation Loss)
3. **Bayesian Hyperparameter Search Convergence**

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# Load experiment ledger
runs = []
with open("artifacts/experiment_ledger.jsonl", "r") as f:
    for line in f:
        if line.strip():
            runs.append(json.loads(line))

ledger_df = pd.DataFrame([
    {
        "Run": r["experiment_id"],
        "ROC_AUC": r["metrics"]["oof_roc_auc"],
        "PR_AUC": r["metrics"]["oof_pr_auc"],
        "Train_Loss": r["metrics"]["final_train_loss"],
        "Val_Loss": r["metrics"]["final_val_loss"],
        "Depth": r["hyperparameters"].get("depth"),
        "Learning_Rate": r["hyperparameters"].get("learning_rate"),
        "L2_Leaf_Reg": r["hyperparameters"].get("l2_leaf_reg"),
    }
    for r in runs
])

print("=== EXPERIMENT LEDGER SUMMARY ===")
display(ledger_df)

# Plot Metric Trajectories
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ledger_df["Run"], ledger_df["ROC_AUC"], marker="o", label="OOF ROC-AUC", color="#1f77b4", lw=2)
axes[0].plot(ledger_df["Run"], ledger_df["PR_AUC"], marker="s", label="OOF PR-AUC", color="#2ca02c", lw=2)
axes[0].set_title("Validation Metrics Progression (15 Runs)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Experiment Run")
axes[0].set_ylabel("Metric Score")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

axes[1].plot(ledger_df["Run"], ledger_df["Train_Loss"], marker="o", label="Final Train Loss", color="#ff7f0e", lw=2)
axes[1].plot(ledger_df["Run"], ledger_df["Val_Loss"], marker="x", label="Final Val Loss", color="#d62728", lw=2)
axes[1].set_title("Generalization Gap & Loss Convergence", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Experiment Run")
axes[1].set_ylabel("Log Loss")
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 8: Evaluate Sealed Holdout & Export Competition Submission

Following the completion of Experiment 15, unseal the 20% holdout set and generate `artifacts/submission.csv`.

In [ ]:
!python scripts/evaluate_holdout.py
!python scripts/export_submission.py

## Step 9: Download Artifacts or Persist to Google Drive

Bundle the experiment ledger, champion model, holdout metrics, and submission file into a downloadable archive.

In [ ]:
import shutil
from pathlib import Path

# Archive the artifacts folder
shutil.make_archive("aads_artifacts", "zip", "artifacts")
print("Artifacts archived to aads_artifacts.zip")

# Optional: Trigger browser download in Google Colab
try:
    from google.colab import files
    files.download("aads_artifacts.zip")
    files.download("artifacts/submission.csv")
except Exception as e:
    print(f"Note: If running locally or download prompt blocked, file is available at aads_artifacts.zip: {e}")